# Delta-Hedging Simulation

This notebook simulates a delta hedge over time. It picks one European option, simulates stock paths, rebalances the stock hedge at each step, tracks the cash account, and measures the final hedging error across many paths.

## Hedge mechanics

A long option has stock exposure measured by Delta. To hedge a long call, the simulator shorts Delta shares. At each rebalance date, it recalculates the option value and Delta, then adjusts the stock hedge.

The cash account tracks:

- buying the option at the start
- proceeds from shorting stock
- costs of buying or selling stock during rebalancing
- interest on cash
- transaction costs

At expiry, the option value becomes the payoff. The final hedge portfolio value plus the option payoff gives the hedging error.

## Why hedging error is a distribution

A single simulated path gives one hedging error. Many paths give a distribution of errors. That distribution is more useful because hedging performance depends on the path the stock takes, not only the starting and ending prices.

High-Gamma options tend to create more rebalancing pressure because Delta changes quickly when the stock moves.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.hedging import (
    HedgeContract,
    build_hedging_figures,
    hedging_error_summary,
    save_hedging_outputs,
    simulate_delta_hedge_many_paths,
)
from src.simulation import simulate_gbm_paths

## Simulation inputs

In [ ]:
starting_price = 100.0
strike = 100.0
risk_free_rate = 0.04
volatility = 0.25
maturity_years = 30 / 365.25

steps = 30
num_paths = 5_000
random_seed = 42
transaction_cost_bps = 5.0

## Simulate stock paths

In [ ]:
paths = simulate_gbm_paths(
    starting_price=starting_price,
    drift=risk_free_rate,
    volatility=volatility,
    time_horizon=maturity_years,
    steps=steps,
    num_paths=num_paths,
    random_seed=random_seed,
)

paths.head()

## Run delta hedge simulation

In [ ]:
contract = HedgeContract(
    option_type="call",
    strike=strike,
    maturity_years=maturity_years,
    risk_free_rate=risk_free_rate,
    volatility=volatility,
    position=1,
)

hedge_results, hedge_details = simulate_delta_hedge_many_paths(
    paths=paths,
    contract=contract,
    transaction_cost_bps=transaction_cost_bps,
    store_path_details=False,
)

hedge_results.head()

## Hedging error summary

In [ ]:
summary = hedging_error_summary(hedge_results)
summary

## Save outputs

In [ ]:
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"

saved_tables = save_hedging_outputs(
    hedge_results=hedge_results,
    summary=summary,
    output_dir=processed_dir,
)

saved_figures = build_hedging_figures(
    hedge_results=hedge_results,
    output_dir=figures_dir,
)

saved_tables, saved_figures

## Key outputs

The most important values are:

- mean hedging error
- standard deviation of hedging error
- 5th percentile hedging error
- 95th percentile hedging error
- average transaction cost

These values summarize the full hedging error distribution.

## Interpretation

Even with the same option and the same volatility assumption, different paths create different hedging errors. The hedge is rebalanced discretely, not continuously, and every rebalance can create transaction costs. This is why the simulator measures a distribution of outcomes instead of claiming the hedge is perfect.